# Lab 2: GPT from scratch

In this lab, you will dive into the inner workings of the GPT architecture. You will walk through a complete implementation of the architecture in PyTorch, instantiate this implementation with pre-trained weights, and put the resulting model to the test by generating text. At the end of this lab, you will understand the building blocks of the GPT architecture and how they are connected.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

In [1]:
from dataclasses import dataclass

import torch
import torch.nn as nn

## Part 1: GPT architecture

GPT-2 was first described by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). To faithfully implement the model, one needs to also read the earlier paper by [Radford et al. (2018)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf). Another important source of information is the code released by OpenAI, which is available on GitHub ([link](https://github.com/openai/gpt-2)).

The GPT architecture is made up of a stack of Transformer blocks. Each block has two main parts: one handles multi-head self-attention, and the other is a feed-forward network. Before these parts do their work, their input undergoes layer normalisation, and residual connections are added to help the model learn more effectively. The input to the architecture is a sequence of token IDs; these are turned into embeddings and augmented with information about the absolute position of each token in the sequence. The output layer converts the internal representations into logit scores for every token in the vocabulary.

### Model configuration

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) present four increasingly larger GPT models based on the same architecture. Here, we will implement the smallest of these, characterised by the following hyperparameters:

In [2]:
@dataclass
class Config:
    n_vocab: int = 50_257
    n_ctx: int = 1024
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12

#### 🧩 Task 2.01: Model configuration

Explain the purpose of these hyperparameters. In particular, where does the number 50,257 come from?

### 🧩 Task 2.01 – Answer

The configuration defines the architecture and capacity of the GPT-2 language model.

- **`n_vocab = 50,257`**  
  The size of the model’s vocabulary, i.e. the number of distinct tokens the model can represent and predict.  
  The value 50,257 comes from the pretrained GPT-2 tokenizer, which uses Byte Pair Encoding (BPE) to construct a fixed vocabulary of subword tokens learned from the training corpus. These tokens include full words, subword fragments, punctuation, and special symbols.

- **`n_ctx = 1024`**  
  The maximum context length of the model, meaning that GPT-2 can condition its predictions on at most 1024 previous tokens.

- **`n_embd = 768`**  
  The dimensionality of the token embeddings and hidden representations used throughout the model.

- **`n_head = 12`**  
  The number of attention heads in each Transformer layer. Multi-head attention allows the model to attend to different aspects of the context in parallel.

- **`n_layer = 12`**  
  The number of stacked Transformer decoder layers in the model.

Together, these hyperparameters specify the standard GPT-2 (small) architecture used in this lab.

---

### GELU activation function

We start by implementing the feed-forward network. This is a standard two-layer network with a Gaussian Error Linear Unit (GELU) activation function ([Hendrycks and Gimpel, 2016](https://doi.org/10.48550/arXiv.1606.08415)).

The GELU is a smooth version of the rectified linear unit (ReLU) that weights inputs by their value under the cumulative distribution function of the standard Gaussian. This function is commonly denoted by $\Phi$. For example, $\text{GELU}(0{.}5) = 0{.}5 \cdot \Phi(0{.}5) \approx 0{.}5 \cdot 0{.}6915 = 0{.}3457$ because approximately 69.15% of normally distributed data lies to the left of $0{.}5$.

When GPT-2 was released, computing the GELU exactly was expensive, and the released code therefore used an approximation originally presented by [Page (1977)](https://doi.org/10.2307/2346872). We follow suit here, as we want to create a replica of the original model. However, it is worth mentioning that PyTorch now offers an exact implementation of the GELU so fast that using an approximation is unnecessary.

In [ ]:
def gelu(x):
    # This is a smooth approximation of GELU that prevents dead neurons, 
    # by allowing a small, non linear flow of negative values.
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x**3)))

#### 🎓 Task 2.02: Mathematical properties of the GELU

Find the minimal output value of the (approximated) GELU and the input value for which it yields that output. Use a service such as [WolframAlpha](https://www.wolframalpha.com/) for the necessary derivations. What are the main differences between the GELU and the ReLU?

### 🎓 Task 2.02 – Answer

By differentiating this function and solving for stationary points numerically, we find that the GELU has a global minimum at approximately  
approx -0.752,
where the output value is  
approx -0.170.

**Differences between GELU and ReLU**

ReLU keeps positive input values unchanged and sets all negative values to zero, which means negative inputs are completely discarded. GELU, on the other hand, smoothly reduces negative values instead of cutting them off, allowing small negative outputs.

Another key difference is smoothness. ReLU is not differentiable at zero and has zero gradients for negative inputs, while GELU is smooth and differentiable everywhere. This results in better gradient flow and more stable optimisation.

Because of this smoother behaviour, GELU is commonly used in Transformer-based models such as GPT-2.

---

### Feed-forward network

Next, here is the code for the feed-forward network. Note that we follow the released code and use the name **multi-layer perceptron (MLP)** rather than “feed-forward network”.

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Expands the embedding size 4x to create a higher 
        # dimensional space for learning features.
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        # Projects the data back to the original size so it
        # can be added to the residual path.
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        # x: [batch_size, seq_len, n_embd]
        # Extracts dimensions to ensure the data
        # matches the expected transformer architecture.
        batch_size, seq_len, n_embd = x.shape

        # c_fc expands the embedding dimension
        # This creates a higher dimensional space to 
        # allow the model to learn more complex features.
        # input:  [batch_size, seq_len, n_embd]
        # output: [batch_size, seq_len, 4 * n_embd]
        x = self.c_fc(x)

        # GELU is applied element-wise and does not change the shape
        # This adds non linearity, enabling the network to solve
        # non linear problems rather than just simple math.
        # output: [batch_size, seq_len, 4 * n_embd]
        x = gelu(x)

        # c_proj projects back to the original embedding dimension
        # This compresses the data back so it can be added to the
        # original input via a residual connection.
        # input:  [batch_size, seq_len, 4 * n_embd]
        # output: [batch_size, seq_len, n_embd]
        x = self.c_proj(x)

        # output: [batch_size, seq_len, n_embd]
        # Returns the processed features to the
        # next layer in the transformer stack.
        return x

#### 🎓 Task 2.03: Shape annotations

One of the most common errors in deep learning is a mismatch in tensor dimensions. To avoid this, it is good practice to annotate PyTorch code with shapes. For example, suppose you are given the following code:

In [5]:
f = nn.Linear(5, 7)
x = torch.rand(2, 3, 5)
y = f(x)

The annotation of this code with shapes would look as follows:

In [ ]:
# Linear layer that transforms the last dimension from size 5 to size 7
f = nn.Linear(5, 7)
# f is a module (function), not a data tensor, so it doesn't have a shape attribute

# Create a random input tensor with dimensions: [batch, sequence, features]
x = torch.rand(2, 3, 5)
# Input shape of x: [2, 3, 5]

# Apply the linear transformation to the input
y = f(x)
# Output shape: [2, 3, 7]
# Notice only the last dimension (the features) changed


Annotate the shapes in the `forward()` method of the feed-forward network. Instead of using actual numbers, refer to dimension sizes by symbolic names such as `n_embd`, `batch_size` (number of samples in a batch of input data) and `seq_len` (length of an input sequence). You can introduce additional names and other notation you find useful. Make your annotations as detailed as you need them to explain how the shapes change from one line to the next.

### 🎓 Task 2.03 – Answer

See the forward method above where we added comments for the code.

---

### Causal mask

Our next goal is to implement the core of the GPT architecture: the multi-head attention mechanism.

Recall that the attention mechanism in the Transformer decoder must be restricted to attending only to previously generated tokens. This type of attention is also called **causal attention**. In practice, we implement it through a masking technique that sets the post-softmax attention weights of future tokens to zero. The following utility function implements such a mask:

In [ ]:
def make_causal_mask(n):
    # Creates an upper triangular matrix of negative infinity to mask future tokens.
    # This makes sure the model only attends to past and current positions during training.
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

#### 🧩 Task 2.04: Causal mask

Have a close look at the following code and run it to see the result. What are the shapes of `x` and `mask`? Given that the shapes are different, why does the addition operation in the last line not raise an error? What is the shape of the result?

How does the addition operation implement masking? (Recall that the attention scores are normalised using the softmax function.)

In [ ]:
# Random tensor of shape (batch=1, channels=2, 3x3 grid)
x = torch.rand(1, 2, 3, 3)

# Create a 5x5 causal (no looking ahead) mask
mask = make_causal_mask(5)

# Use the top left 3x3 of the mask and add it to x (broadcasted)
x + mask[:3, :3]

tensor([[[[0.0635,   -inf,   -inf],
          [0.6721, 0.0349,   -inf],
          [0.6761, 0.6064, 0.4698]],

         [[0.8823,   -inf,   -inf],
          [0.2552, 0.7285,   -inf],
          [0.2986, 0.9612, 0.8597]]]])

### 🧩 Task 2.04 – Answer

**1. What are the shapes of `x` and `mask`?**

The tensor `x` is created using `torch.rand(1, 2, 3, 3)` and therefore has shape  
\[
[1, 2, 3, 3].
\]
The causal mask is created using `make_causal_mask(5)`, which returns a tensor of shape  
\[
[5, 5].
\]
After slicing with `mask[:3, :3]`, the mask used in the addition has shape  
\[
[3, 3].
\]

**2. Given that the shapes are different, why does the addition operation in the last line not raise an error?**

The addition does not raise an error because PyTorch applies broadcasting. The `[3, 3]` mask is automatically expanded along the batch and head dimensions to match the shape of `x`, allowing element-wise addition.

**3. What is the shape of the result?**

After broadcasting and addition, the resulting tensor has the same shape as `x`, namely  
\[
[1, 2, 3, 3].
\]

**4. How does the addition operation implement masking?**

The causal mask contains `0` for allowed positions and `-∞` for positions corresponding to future tokens. By adding the mask to the attention scores before applying the softmax function, the scores of masked positions become `-∞`. Since `exp(-∞) = 0`, the softmax assigns zero probability to these positions. This prevents tokens from attending to future tokens and enforces causal (autoregressive) attention.

---

### Attention mechanism

Here is the code for the multi-head attention mechanism:

In [ ]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0 # Verify divisibility, ensures the embedding can be split equally into heads
        self.n_head = config.n_head # Store head count, required for reshaping tensors during the forward pass
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3) # Single linear projection, computes Q, K, and V in one matrix multiplication
        self.c_proj = nn.Linear(config.n_embd, config.n_embd) # Output projection, mixes information from all heads back into the residual stream
        self.register_buffer("mask", make_causal_mask(config.n_ctx), persistent=False) # Store the mask, ensures it is moved to the correct device

    def forward(self, x):
        # x: [batch_size, seq_len, n_embd]
        batch_size, seq_len, n_embd = x.shape # Extract dimensions, provides the dynamic shapes needed for reshaping operations
        head_embd = n_embd // self.n_head # Calculate per head size, defines the dimensionality of the subspace each head attends to

        # Linear projection to queries, keys, and values
        # self.c_attn(x): [batch_size, seq_len, 3 * n_embd]
        q, k, v = self.c_attn(x).chunk(3, dim=-1) # Split the projection, separates the combined QKV tensor into individual components
        # q, k, v: [batch_size, seq_len, n_embd]
        # The weights are broadcasted over the batch_size and seq_len dimensions.

        # Reshape to separate attention heads
        q = q.view(batch_size, seq_len, self.n_head, head_embd) # Isolate heads, creates a dedicated dimension for parallel multi head processing
        k = k.view(batch_size, seq_len, self.n_head, head_embd) # Isolate heads, separates the key features for each specific head
        v = v.view(batch_size, seq_len, self.n_head, head_embd) # Isolate heads, separates the value features for each specific head
        # q, k, v: [batch_size, seq_len, n_head, head_embd]

        # Transpose to put head dimension before sequence length
        q = q.transpose(-2, -3) # Reorder the dimensions, aligns the tensor for batch matrix multiplication across heads
        k = k.transpose(-2, -3) # Reorder the dimensions, positions the head dimension as a batch like index for efficiency
        v = v.transpose(-2, -3) # Reorder the dimensions, ensures the sequence dimension is in the correct spot for the weighted sum
        # q, k, v: [batch_size, n_head, seq_len, head_embd]

        # Scaled dot product attention scores
        x = q @ k.transpose(-1, -2) # Compute dot product, measures similarity between all query and key pairs in the sequence
        # The leading [B, H] dimensions are broadcasted as batch dimensions.
        # x: [batch_size, n_head, seq_len, seq_len]

        # Scale by sqrt(head_embd)
        # Broadcasting is used to divide by a scalar
        x = x / head_embd**0.5 # Apply scaling, keeps the variance of the scores unit level to prevent softmax saturation

        # Add causal mask
        # self.mask[:seq_len, :seq_len]: [seq_len, seq_len]
        # Broadcast to [batch_size, n_head, seq_len, seq_len]
        x = x + self.mask[:seq_len, :seq_len] # Apply masking, enforces the autoregressive constraint so tokens don't attend to future tokens

        # Softmax over key dimension
        x = torch.softmax(x, dim=-1) # Normalize scores, converts similarity scores into a probability distribution over the sequence
        # x: [batch_size, n_head, seq_len, seq_len]

        # Apply attention weights to values
        x = x @ v # Weighted sum, aggregates information from the values based on the computed attention weights
        # x: [batch_size, n_head, seq_len, head_embd]

        # Transpose back and merge heads
        x = x.transpose(-2, -3).contiguous() # Restore original order, moves the sequence dimension back to the primary position
        # x: [batch_size, seq_len, n_head, head_embd]

        x = x.view(batch_size, seq_len, n_embd) # Concatenate heads, merges the output of all parallel heads back into the embedding dimension
        # x: [batch_size, seq_len, n_embd]

        # Final linear projection
        x = self.c_proj(x) # Project output, allows the model to learn a final transformation of the concatenated head outputs
        # x: [batch_size, seq_len, n_embd]
        return x # Output results, passes the context aware embeddings to the next layer

#### 🎓 Task 2.05: Multi-head attention

Trace the input `x` through the `forward()` method line by line and annotate the shapes of all tensor variables. Identify all lines that rely on broadcasting.

### 🎓 Task 2.05 – Answer

See the forward method above where we added comments for the shapes of all tensor variables and also the 4 lines we found that rely on broadcasting which are:
1. q, k, v = self.c_attn(x).chunk(3, dim=-1)
2. x = q @ k.transpose(-1, -2)
3. x = x / head_embd**0.5
4. x = x + self.mask[:seq_len, :seq_len]

---

### Layer normalisation

As mentioned above, the inputs to both the feed-forward network and the multi-head attention mechanism undergo **layer normalisation**. This normalises the inputs to have zero mean and unit variance across the activations. [Ba et al. (2016)](https://doi.org/10.48550/arXiv.1607.06450) introduce two trainable parameters (called $\gamma$ and $\beta$ in the paper) that allow the network to learn an appropriate scale and shift for the normalised values.

We implement layer normalisation as follows:

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.g = nn.Parameter(torch.ones(config.n_embd)) # Gain parameter, a learnable weight that allows the model to rescale the normalized output
        self.b = nn.Parameter(torch.zeros(config.n_embd)) # Bias parameter, a learnable offset that allows the model to shift the normalized output

    def forward(self, x):
        # x: [batch_size, seq_len, n_embd]
        mean = x.mean(dim=-1, keepdim=True) # Compute feature mean, calculates the average across the embedding dimension for each token
        variance = x.var(unbiased=False, dim=-1, keepdim=True) # Compute feature variance, measures the spread of values to prepare for scaling
        
        # return: [batch_size, seq_len, n_embd]
        # (x - mean) / sqrt(variance + epsilon) performs the standardization
        # self.g and self.b use broadcasting to apply the same gain/bias to every token in the batch
        return self.g * (x - mean) / torch.sqrt(variance + 1e-05) + self.b # Normalize and transform, stabilizes hidden state dynamics and improves training convergence

#### 🧩 Task 2.06: Layer normalisation

What is the relevance of the `keepdim=True` keyword argument in the `mean()` and `var()` functions? What would happen if we omitted it?

What is the relevance of the constant 1e-05? What could happen if we omitted it?

### 🧩 Task 2.06 – Answer

The `keepdim=True` argument in the `mean()` and `var()` functions ensures that the reduced dimension is retained with size 1. Since layer normalisation computes statistics over the last dimension (`n_embd`), using `keepdim=True` results in tensors of shape `[batch_size, seq_len, 1]`. This allows the mean and variance to be broadcasted correctly when subtracting them from the input tensor `x`. If `keepdim=True` were omitted, the resulting tensors would have shape `[batch_size, seq_len]`, and the subsequent operations would either fail or require additional reshaping.

The constant `1e-05` is added for numerical stability. It prevents division by zero or very small values when computing the standard deviation. If this constant were omitted and the variance became zero or extremely small, the normalisation could produce infinite or undefined values, leading to unstable training or numerical errors.

---

### Decoder block

We now combine the feed-forward network, the multi-head attention mechanism and the layer normalisation into a decoder block.

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config) # First normalization, prepares input for the attention mechanism
        self.attn = Attention(config) # Multi head attention, allows tokens to communicate and exchange information
        self.ln_2 = LayerNorm(config) # Second normalization, stabilizes the signal before the feed forward expansion
        self.mlp = MLP(config) # Feed forward network, processes information within each token independently

    def forward(self, x):
        # x = x + ... represents a residual connection (a skip connection)
        x = x + self.attn(self.ln_1(x)) # Apply attention, pre norm setup helps gradients flow better during training
        x = x + self.mlp(self.ln_2(x)) # Apply MLP, adds non linear complexity and projects back to the residual stream
        return x # Return updated hidden states, provides the refined representation to the next block

#### 🎓 Task 2.07: Pre-norm and post-norm architectures

The original Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) is a “post-norm architecture”, where the normalisation is applied **after** each residual block. In contrast, GPT-2 is a “pre-norm architecture”, where the normalisation is applied **before**. Find the passage in Section&nbsp;2.3 of [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) that reports on this modification.

[Xiong et al. (2020)](https://arxiv.org/pdf/2002.04745) compare pre-norm and post-norm architectures empirically. Read the abstract of their paper and summarise their main findings. According to these findings, what are the benefits of the pre-norm architecture?

### 🎓 Task 2.07 – Answer

[Xiong et al. (2020)](https://arxiv.org/pdf/2002.04745) compare pre-norm and post-norm transformer architectures and show that post-norm models are harder to train, especially when the model becomes deep. In post-norm transformers, training is often unstable and requires careful learning-rate warmup to avoid gradient problems.

In contrast pre-norm transformers are much more stable during training. The authors show that pre-norm models have better gradient flow, do not rely as heavily on learning-rate warm-up, and converge faster, while reaching similar final performance.

According to these findings, the main benefits of the pre-norm architecture are improved training stability, easier optimisation and better scalability to deeper transformer models.

---

### Model

We now have almost all components in place to complete the implementation of the GPT-2 model. The only thing  missing are the position embeddings. These simply associate an embedding vector with every position in the context window. To set them up, we first define another utility function:

In [ ]:
def make_positions(n):
    # Create position indices [0, 1, ..., n-1], this is used for positional encoding
    return torch.arange(n, dtype=torch.long)

We then code the complete model as follows:

In [ ]:
class Model(nn.Module): 
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Token embedding: maps token IDs to vectors
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)

        # Positional embedding: encodes token positions
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)

        # Stack of transformer blocks
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))

        # Final layer normalization
        self.ln_f = LayerNorm(config)

        # Projection from hidden states to vocabulary logits
        self.lm_head = nn.Linear(config.n_embd, config.n_vocab, bias=False)

        # Precomputed position indices buffer
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len = x.shape  # x is [batch_size, seq_len] of token IDs
        wte = self.wte(x)  # Token embeddings → [batch_size, seq_len, n_embd]
        wpe = self.wpe(self.pos[:seq_len])  # Positional embeddings → [seq_len, n_embd]
        x = wte + wpe  # Combine token and positional information
        x = self.h(x)  # Pass through transformer layers
        x = self.ln_f(x)  # Normalize final hidden states
        x = self.lm_head(x)  # Convert to vocabulary logits
        return x  # Output shape: [batch_size, seq_len, n_vocab]

#### 🧩 Task 2.08: Buffers

Our implementation registers the vector of positions as a buffer. (Earlier, we also registered the causal mask as a buffer.) Consult the PyTorch documentation to determine the benefits of registering a tensor as a buffer, in contrast to computing it in the `forward()` method.

### 🧩 Task 2.08 – Answer

Registering a tensor as a buffer makes it part of the model without making it a trainable parameter. Buffers are automatically moved to the correct device when the model is moved between CPU and GPU, and they are saved and loaded together with the model.

By registering the position indices and the causal mask as buffers, they only need to be created once and can be reused in every forward pass. This is more efficient than recomputing them inside the `forward()` method and avoids device-related errors.

If these tensors were created inside `forward()`, they would be recomputed every time and would require extra care to ensure they are placed on the correct device.

---

#### 🎓 Task 2.09: Number of trainable parameters

The model we have implemented is the smallest one presented by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). But how many trainable parameters exactly does it have? Interestingly, the number originally reported by the authors is wrong! What number did they report?

Your task is to write code to compute the number of parameters yourself. This should only take 1–3 lines of code. What number do you get when you apply this code to a fresh model instance?

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) followed the original Transformers paper ([Vaswani et al., 2017](https://doi.org/10.48550/arXiv.1706.03762)) and shared the trainable weights between the token embedding and the final linear layer. Implement this weight sharing strategy. (Hint: This only requires one line of code.) Then, re-compute the number of trainable parameters for the modified model. What number do you get now? How large is the reduction caused by the weight sharing?

In [14]:
# Create a new model instance
model = Model(Config())

# Count trainable parameters
params_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Parameters before weight tying:", params_before)

# Apply weight tying
model.lm_head.weight = model.wte.weight

# Count trainable parameters
params_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Parameters after weight tying:", params_after)

# Reduction due to weight tying
print("Reduction:", params_before - params_after)

Parameters before weight tying: 163037184
Parameters after weight tying: 124439808
Reduction: 38597376


### 🎓 Task 2.09 – Answer

The number of trainable parameters in the model can be computed by summing the sizes of all trainable tensors. Applying this to a freshly initialised model yields approximately 163 million parameters.

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) report that the smallest GPT-2 model has 117 million parameters. This number is incorrect, the correct parameter count is approximately 124 million.

The discrepancy arises because GPT-2 shares the weights between the token embedding layer and the final linear output layer. After implementing this weight sharing, the number of trainable parameters in the model is reduced to approximately 124 million.

The reduction caused by weight sharing is about 38.6 million parameters, corresponding to the size of the vocabulary embedding matrix.

---

## Part 2: Load pre-trained weights

Now that you have a complete implementation of the GPT-2 model in place, you can instantiate it by loading the pre-trained weights released by OpenAI. These weights were originally provided in the TensorFlow format. For this lab, we have re-packaged them as a single file in NumPy’s `.npz` archive format. We can load it as follows:

In [ ]:
import numpy as np

# Load saved pretrained weights
pretrained = np.load("gpt-2-pretrained.npz", allow_pickle=True)

The result `pretrained` is a dictionary mapping names to NumPy arrays. When you print the names, you will see that they correspond to the attributes of our network modules, even though the names differ. For example, the array `h0.attn.c_attn.b` holds the biases (`b`) of the `c_attn` linear layer in the attention mechanism (`attn`) of the first transformer block (`h0`).

#### 🎓 Task 2.10: Load pre-trained weights

Create a model from the pre-trained weights. To do this, you need to instantiate a fresh model and write the contents of each array from the `npz` archive with the pre-trained weights into the corresponding tensor. To make this a bit easier, here is a utility function that copies data from a NumPy array `source` to a PyTorch tensor `target`:

In [ ]:
def copy_weights(source: np.ndarray, target: torch.Tensor):
    # Make sure the array and the tensor have identical shapes
    assert source.shape == target.shape

    # Disable gradient tracking since we are manually assigning weights
    with torch.no_grad():
        # Convert array to a float32 tensor and copy data into target tensor
        target.copy_(torch.tensor(source, dtype=torch.float32))

You can start from this skeleton code:

In [ ]:
def from_pretrained() -> Model:
    # Create a fresh model using the default config
    model = Model(Config())

    # Load the checkpoint containing the pretrained GPT-2 weights
    pretrained = np.load("gpt-2-pretrained.npz", allow_pickle=True)

    # Embeddings
    # Token embeddings: map vocabulary IDs to vectors learned during pretraining
    copy_weights(pretrained["wte"], model.wte.weight)

    # Positional embeddings: encode token position information for the transformer
    copy_weights(pretrained["wpe"], model.wpe.weight)

    # Transformer Blocks
    for i in range(model.config.n_layer):
        block = model.h[i] # Select the i-th transformer block

        # Attention projections
        # Transpose is needed because weights are stored as (in, out)
        # while PyTorch Linear expects (out, in)
        copy_weights(pretrained[f"h{i}.attn.c_attn.w"].T, block.attn.c_attn.weight)
        copy_weights(pretrained[f"h{i}.attn.c_attn.b"],   block.attn.c_attn.bias)

        # Output projection of the attention module
        copy_weights(pretrained[f"h{i}.attn.c_proj.w"].T, block.attn.c_proj.weight)
        copy_weights(pretrained[f"h{i}.attn.c_proj.b"],   block.attn.c_proj.bias)

        # Feed-forward (MLP) first layer: expands hidden dimension
        copy_weights(pretrained[f"h{i}.mlp.c_fc.w"].T, block.mlp.c_fc.weight)
        copy_weights(pretrained[f"h{i}.mlp.c_fc.b"],   block.mlp.c_fc.bias)

        # Feed-forward projection back to model dimension
        copy_weights(pretrained[f"h{i}.mlp.c_proj.w"].T, block.mlp.c_proj.weight)
        copy_weights(pretrained[f"h{i}.mlp.c_proj.b"],   block.mlp.c_proj.bias)

        # LayerNorm parameters (scale=g, bias=b) for training stability
        copy_weights(pretrained[f"h{i}.ln_1.g"], block.ln_1.g)
        copy_weights(pretrained[f"h{i}.ln_1.b"], block.ln_1.b)
        copy_weights(pretrained[f"h{i}.ln_2.g"], block.ln_2.g)
        copy_weights(pretrained[f"h{i}.ln_2.b"], block.ln_2.b)

    # Final normalization before logits
    copy_weights(pretrained["ln_f.g"], model.ln_f.g)
    copy_weights(pretrained["ln_f.b"], model.ln_f.b)

    # Output head
    # GPT-2 ties output weights to token embeddings, so we reuse wte here
    copy_weights(pretrained["wte"], model.lm_head.weight)

    return model # Return model now populated with pretrained weights

**Important:** One technical detail to note is that PyTorch stores the weights of linear layers in a transposed form. For example, a linear layer created as `nn.Linear(2, 3)` has a weight matrix of shape [3, 2].

## Part 3: Put the model to use

In the third and final part of this lab, you will use the pre-trained model to generate text and evaluate it on a standard benchmark.

### Sampling-based text generation

The easiest way to generate text with a language model is by using a **greedy approach**. This method works by creating text one token at a time. At each step, the model takes the previously generated text (called the **context**) as input and adds the token with the highest output logit as a new token. The code in the next cell defines a function `generate()` that forms the core of a greedy generator:

In [18]:
import torch
import torch.nn.functional as F

def generate(model, context, context_size=1024, n_tokens=20, temperature=1, top_k=None):
    for _ in range(n_tokens):
        # Limit the context if it goes over the models max context size
        context_cond = context[:, -context_size:]
        with torch.no_grad():
            # Get the logits for the last token
            logits = model(context_cond)[:, -1, :]
            
            # 1. Temperature scaling
            # Higher temp means more random, and lower temp means more predictable
            logits = logits / temperature
            
            # 2. Top-k sampling
            if top_k is not None:
                # Find values and indexes of top k logits
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                # Mask out any values less than the k-th value
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            # 3. Change logits to probabilities aka softmax
            probs = F.softmax(logits, dim=-1)
            
            # 4. Sample from the categorical distribution
            next_idx = torch.multinomial(probs, num_samples=1)
            
            # Add it to the context
            context = torch.cat([context, next_idx], dim=-1)
            
    return context

To use this function with an actual text input, you need a tokeniser to first encode the text into a vector of token IDs, and later decode the generated `context` into new text. The reference implementation of the GPT-2 tokeniser is in the library `tiktoken`. The code in the next cell sets up the tokeniser, loads the pretrained model from Task&nbsp;2.10, and then defines a helper function that handles the encoding and decoding.

In [ ]:
import sys, site

print(sys.executable)
print(site.getsitepackages())

import tiktoken

# Load the GPT-2 tokenizer so text to token IDs match the pretrained weights
tokenizer = tiktoken.get_encoding("gpt2")

# Load model with pretrained weights
model = from_pretrained()

def generate_helper(text, context_size=1024, n_tokens=20):
    # Encode input string into token IDs and wrap in a batch dimension
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)

    # Run autoregressive generation to append new tokens
    context = generate(model, context, context_size=context_size, n_tokens=n_tokens)

    # Decode tokens back into human readable text
    return tokenizer.decode(context[0].tolist())

/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv311/bin/python
['/Users/dennisjohansson/Skola/Programering/VT4/TDDE09/tdde09_labs/.venv311/lib/python3.11/site-packages']


You can use this helper function to generate text as follows:

In [20]:
generate_helper("Linköping University is")

"Linköping University is linked by young people to the World Bank's Corruption and Group Watch Index, the books of which cover"

**Tip:** If you did not manage to complete Task&nbsp;2.10, you can still work on this task by using a pretrained GPT-2 model from [Hugging Face](https://huggingface.co/openai-community/gpt2). The next code cell shows how you would instantiate this model. Note that you may have to first install the `transformers` library.

In [21]:
# from transformers import GPT2LMHeadModel
# model = GPT2LMHeadModel.from_pretrained("gpt2")
# logits = model(context).logits[:, -1, :]

#### 🎓 Task 2.11: Sampling-based text generation

 The greedy approach to text generation is not very interesting for practical applications because it always chooses the most likely token, leading to predictable and less creative results. Your task is to modify the code for the `generate()` function to use a **sampling-based approach** instead. In this approach, the next token is chosen randomly based on the probabilities assigned by the model (softmax-normalised logits), treating them as a categorical distribution over the token vocabulary. Additionally, your code should include two common techniques to improve sampling:
 
 * **temperature scaling**, which lets the user control the randomness of the sampling
 * **top-$k$ sampling**, which limits the sampling to the top-$k$ most likely tokens, ignoring less probable ones

### Evaluating the pretrained model

If you have experimented with your pretrained GPT-2 model, you will have noticed that its ability to generate useful text is somewhat limited. By today’s standards, GPT-2 is a small model with modest capabilities. However, it can still be helpful for certain tasks, such as text autocompletion, generating filler text, or answering simple questions. To rigourosly evaluate language models, researchers often use standard benchmark datasets. Creating these benchmarks is a discipline of its own, and they tend to become increasingly challenging as models continue to improve.

In the final task of this lab, you will evaluate GPT-2’s performance on a small subset of the [HellaSwag dataset](https://rowanzellers.com/hellaswag/), which was published in the same year as GPT-2 itself (2019). HellaSwag is designed to test a model’s ability to perform commonsense reasoning in challenging contexts. Unlike simpler benchmarks, HellaSwag presents scenarios where the correct text completion depends on semantic relationships between events and on world knowledge. This makes it a good choice for assessing the ability of language models to go beyond surface-level patterns and produce meaningful, context-aware predictions.

#### 🎓 Task 2.12: Evaluating the pretrained model

Read the [HellaSwag website](https://rowanzellers.com/hellaswag/) to get some background on the benchmark. How does a sample from the dataset look like? What is an expected prediction? How does the benchmark allow us to score models? What is the random baseline? What is the human performance reported on the task?

The next cell contains code for evaluating your pretrained model on a small sample from HellaSwag. You will also need a tokenizer. The HellaSwag subset is in the file `hellaswag-mini.jsonl`. Inspect that file to understand the format. Next, read the code and explain how it works. Specifically, how does the code compute the score of individual endings? In the call to `cross_entropy()`, why are the tensors sliced in this specific way?

Finally, what overall score does the pretrained GPT-2 model get on this benchmark? How does that score compare to the random baseline and the human performance?

In [ ]:
import json

with open("hellaswag-mini.jsonl") as f:
    n_correct = 0
    n_total = 0
    for line in f:
        # Parse one sample from the dataset
        sample = json.loads(line)

        # Tokenize the context (prefix text)
        # This part is shared across all four endings
        prefix = tokenizer.encode(sample["ctx"])

        ending_scores = []

        # Evaluate each of the four candidate endings
        for i, ending in enumerate(sample["endings"]):
            # Tokenize the ending (with leading space for correct tokenization)
            suffix = tokenizer.encode(" " + ending)

            # Concatenate context + ending into a single input sequence
            context = torch.tensor([prefix + suffix], dtype=torch.long)

            # Disable gradients since this is evaluation only
            with torch.no_grad():
                # Run the model to obtain logits for all tokens
                logits = model(context)

                # Compute cross-entropy loss for this ending only
                # GPT models predict the next token at each position
                # (logits[:, t] predicts token t+1).
                # We slice the tensors so predictions are aligned with the true
                # ending tokens and only the ending contributes to the loss.
                ending_score = torch.nn.functional.cross_entropy(
                    logits[0, -len(suffix) - 1 : -1], context[0, -len(suffix) :]
                )

            # Store (loss, ending index)
            # Lower loss means the ending is more likely according to the model
            ending_scores.append((ending_score, i))

        # Select the ending with the lowest cross-entropy loss
        predicted = min(ending_scores)[1]

        # Compare prediction with the correct label
        n_correct += int(predicted == sample["label"])
        n_total += 1
    
    # Print final accuracy
    print(f"Accuracy: {n_correct / n_total:.2%}")

Accuracy: 30.47%


## 🎓 Task 2.12 – Answer
### How does a sample from the dataset look like?

Each sample in the HellaSwag dataset consists of:
- a **context** (`ctx`) describing a short situation
- **four possible endings** (`endings`)
- a **label** (`label`) indicating which ending (0–3) is correct

For example, a context may describe a person cooking in a kitchen, followed by four different endings, where only one is a plausible continuation of the situation.

### What is an expected prediction? How does the benchmark allow us to score models?

The expected prediction is the **index of the ending that best continues the given context**.

The benchmark scores models using **accuracy**:
- For each sample, the model selects one of the four endings.
- The prediction is correct if it matches the provided label.
- The final score is the proportion of correctly predicted samples.

### What is the random baseline?

Since there are four possible endings for each sample, a random guess achieves:

- **25% accuracy**

This is the random baseline for the benchmark.

### What is the human performance reported on the task?

Human performance on the HellaSwag benchmark is reported to be around:

- **95–96% accuracy**

This large gap between human and model performance highlights the difficulty of commonsense reasoning for language models.

### How does the code compute the score of individual endings?

For each HellaSwag sample, the model evaluates all four candidate endings separately.
Each ending is appended to the context and passed through the language model.
The score of an ending is computed using **cross-entropy loss**, which measures how
well the model predicts the ending tokens given the context.

The ending with the **lowest cross-entropy loss** is considered the most likely
continuation and is selected as the model’s prediction.


### Why are the tensors sliced in this specific way in `cross_entropy()`?

GPT-style language models predict the **next token** at each position in the input.
Therefore, predictions at position *t* correspond to the true token at position *t+1*.

To correctly align predictions and targets and to score **only the ending**:
- `logits[0, -len(suffix)-1 : -1]` selects the model’s predictions for the ending tokens
- `context[0, -len(suffix):]` selects the true ending tokens

This slicing ensures that:
- the context tokens do not contribute to the loss
- predictions and targets are correctly aligned
- only the likelihood of the ending is evaluated

### Final result

The pretrained GPT-2 model achieves **30.47% accuracy** on HellaSwag.  
This is slightly above the **random baseline of 25%**, but far below **human performance of about 96%**.

---

**🥳 Congratulations on finishing lab&nbsp;2!**